In [ ]:
import pandas as pd
import numpy as np
import requests
import re

In [ ]:
annotated_proteins=pd.read_csv('Re-Run_Seq.csv')
#annotated_proteins.head()

In [ ]:
annotated_proteins_entry_id=annotated_proteins['EntryID' ].tolist()

In [ ]:
import requests
import re
import time

start = time.time()

annotated_protein = {}

def fetch_annotated_protein_dates(protein_id):
    url = f'https://rest.uniprot.org/unisave/{protein_id}?format=txt'
    response = requests.get(url, stream=True)  # Use stream=True to read the content incrementally

    try:
        if response.status_code == 200:
            for line in response.iter_lines(decode_unicode=True):
                # Check for creation date
                if 'integrated into UniProtKB/Swiss-Prot' in line:
                    creation_match = re.search(r'DT\s+(\d{2}-[A-Z]{3}-\d{4}), integrated', line)
                    if creation_match:
                        creation_date = creation_match.group(1)
                        annotated_protein[protein_id] = creation_date
                        return creation_date
        return "Creation date not found"  # Return if no creation date is found in the text
    except Exception as e:
        return f"An error occurred: {e}"

# Example usage
# Assume annotated_proteins_entry_id is a list of protein IDs
for protein_id in annotated_proteins_entry_id[0:20000]:
    dates = fetch_annotated_protein_dates(protein_id)

end = time.time()
print('Total time is', end - start)


KeyboardInterrupt: 

In [ ]:
df=pd.DataFrame(annotated_protein.values(),columns=['First_Release'],index=annotated_protein.keys())

df=df.reset_index()

df.to_csv('Annotated_Proteins_Last_1.0.csv',index=False)